# Train / Validation / Test Split

This notebook creates the evaluation splits for the cold-start project. It loads the
cleaned tables produced by `data_cleaning.ipynb` from `../outputs/` and:

1. selects the new-user and new-item cold-start test groups,
2. creates the standard 70 / 15 / 15 train / validation / test split,
3. builds the sparse-user test group,
4. runs the data-leakage checks,
5. saves all split files to `../outputs/` and shows a summary table.

**Run `data_cleaning.ipynb` first** - it produces `processed_items.csv` and
`processed_interactions.csv`, which this notebook needs.

In [ ]:
# Library
import os
from pathlib import Path

import numpy as np
import pandas as pd

# make pandas output a bit easier to read
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

### Project paths and configuration

All paths are resolved relative to the project root (`netflix-cold-start/`), so the
notebook runs no matter what the kernel's current working directory is. The split
settings live here too.

- cleaned inputs (from `data_cleaning.ipynb`): `../outputs/`
- split outputs: `../outputs/`

In [ ]:
# ---- Project paths (portable - no hard-coded absolute paths) ----

def find_project_root(markers=("data", "notebooks", "outputs")):
    """Locate the project root folder regardless of the working directory.

    Walks upward from the notebook's own location (available in VS Code) or from
    the current working directory until it finds a folder that directly contains
    all of the `markers` subfolders.

    Parameters
    ----------
    markers : tuple of str
        Folder names that must all exist directly inside the project root.

    Returns
    -------
    pathlib.Path
        Absolute path of the project root.
    """
    starts = []
    nb_file = globals().get("__vsc_ipynb_file__")  # set by VS Code's notebook editor
    if nb_file:
        starts.append(Path(nb_file).resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in (start, *start.parents):
            if all((candidate / m).is_dir() for m in markers):
                return candidate
    raise FileNotFoundError(
        "Could not find the project root. Start the kernel inside the "
        "netflix-cold-start folder, or set PROJECT_ROOT manually."
    )

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

# ---- Reproducibility ----
SEED = 42
np.random.seed(SEED)

# ---- Cold-start sampling settings ----
COLD_USER_FRAC = 0.10   # fraction of eligible users held out as new-user cold start
COLD_ITEM_FRAC = 0.10   # fraction of eligible items held out as new-item cold start

### Reusable function definitions

All reusable logic lives in the small functions below, one function per cell. Each takes
its inputs (dataframes, column names, settings) as parameters and returns its result, so
the same functions can be rerun later with different input files, columns, or settings.
The execution cells further down call them in the same order as the original workflow.

In [ ]:
def select_cold_start_users(interactions, frac, seed, min_interactions=2):
    """Sample users to hold out entirely as the new-user cold-start test group.

    Users need at least `min_interactions` interactions to be eligible, so the
    cold-start evaluation is stable. Also prints how many single-interaction
    users exist, since that long tail is exactly what makes cold start hard.

    Parameters
    ----------
    interactions : pandas.DataFrame
        Must contain a "user_id" column.
    frac : float
        Fraction of eligible users to sample.
    seed : int
        Random state for reproducibility.
    min_interactions : int

    Returns
    -------
    set
        The sampled cold-start user ids.
    """
    user_counts = interactions.groupby("user_id").size()
    single_interaction_users = (user_counts == 1).sum()
    print(f"Users with only 1 interaction: {single_interaction_users} "
          f"({single_interaction_users / len(user_counts) * 100:.1f}% of users)")

    # users with >= min_interactions are 'eligible' so the cold-start eval is stable
    eligible_users = user_counts[user_counts >= min_interactions].index.to_series()
    cold_users = set(eligible_users.sample(frac=frac, random_state=seed))
    print(f"Selected {len(cold_users)} cold-start users")
    return cold_users

In [ ]:
def select_cold_start_items(interactions, frac, seed, min_interactions=2):
    """Sample items to hold out as the new-item cold-start test group.

    Movies and books are sampled separately (item ids carry a "movie_" / "book_"
    prefix) so both item types are represented in the cold-start test set.

    Parameters
    ----------
    interactions : pandas.DataFrame
        Must contain an "item_id" column with prefixed ids.
    frac : float
        Fraction of eligible items to sample, per item type.
    seed : int
        Random state for reproducibility.
    min_interactions : int

    Returns
    -------
    set
        The sampled cold-start item ids.
    """
    item_counts = interactions.groupby("item_id").size()
    eligible_items = item_counts[item_counts >= min_interactions].index

    eligible_movie_items = pd.Series([i for i in eligible_items if i.startswith("movie_")])
    eligible_book_items = pd.Series([i for i in eligible_items if i.startswith("book_")])

    cold_items = set(eligible_movie_items.sample(frac=frac, random_state=seed)) | \
                 set(eligible_book_items.sample(frac=frac, random_state=seed))
    print(f"Selected {len(cold_items)} cold-start items "
          f"({sum(i.startswith('movie_') for i in cold_items)} movies, "
          f"{sum(i.startswith('book_') for i in cold_items)} books)")
    return cold_items

In [ ]:
def split_train_val_test(remaining, has_timestamp, seed, train_frac=0.70, val_frac=0.15):
    """Split interactions into train / validation / test sets.

    Chronological when timestamps are usable (train on the past, test on the
    future, which is more realistic), otherwise a reproducible random split.
    The test set gets whatever `train_frac` and `val_frac` leave over.

    Parameters
    ----------
    remaining : pandas.DataFrame
        Interactions not reserved for the cold-start test sets.
    has_timestamp : bool
        Whether a usable "timestamp" column exists.
    seed : int
        Random state for the random split.
    train_frac, val_frac : float

    Returns
    -------
    (train, validation, test) : tuple of pandas.DataFrame
    """
    if has_timestamp:
        # chronological: earliest rows train, middle validation, latest test
        ordered = remaining.sort_values("timestamp").reset_index(drop=True)
        print("Used a CHRONOLOGICAL split (timestamps available).")
    else:
        # reproducible random split
        ordered = remaining.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        print("Used a RANDOM split (no usable timestamps).")

    n = len(ordered)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))
    train = ordered.iloc[:train_end].copy()
    validation = ordered.iloc[train_end:val_end].copy()
    test = ordered.iloc[val_end:].copy()
    return train, validation, test

In [ ]:
def split_summary(df, name, total_interactions):
    """Summarize one interaction split as a dict (for building a comparison table).

    Parameters
    ----------
    df : pandas.DataFrame
        One split (must have user_id, item_id, item_type columns).
    name : str
        Split name shown in the summary row.
    total_interactions : int
        Total interaction count, used for the percentage column.

    Returns
    -------
    dict
    """
    return {
        "dataset": name,
        "rows": len(df),
        "unique_users": df["user_id"].nunique(),
        "unique_items": df["item_id"].nunique(),
        "pct_of_all_interactions": round(len(df) / total_interactions * 100, 2),
        "movies": (df["item_type"] == "movie").sum(),
        "books": (df["item_type"] == "book").sum(),
    }

## Load the cleaned tables

The cleaned item and interaction tables come from `data_cleaning.ipynb`.

In [ ]:
# load the cleaned tables produced by data_cleaning.ipynb
items_path = OUTPUT_DIR / "processed_items.csv"
interactions_path = OUTPUT_DIR / "processed_interactions.csv"

if not items_path.exists():
    raise FileNotFoundError(f"{items_path} not found - run data_cleaning.ipynb first.")
items = pd.read_csv(items_path)
print("Items loaded:", items.shape)

interactions = None
if interactions_path.exists():
    interactions = pd.read_csv(interactions_path)
    print("Interactions loaded:", interactions.shape)
else:
    print("No interactions file found - run data_cleaning.ipynb first if this is wrong.")

# timestamps: data_cleaning.ipynb only keeps this column when it is usable
if interactions is not None and "timestamp" in interactions.columns:
    interactions["timestamp"] = pd.to_datetime(interactions["timestamp"], errors="coerce")
    HAS_TIMESTAMP = interactions["timestamp"].notna().mean() > 0.9
else:
    HAS_TIMESTAMP = False
print("Using timestamps for splitting:", HAS_TIMESTAMP)

## 14. Define the cold-start evaluation strategy

A completely random split is **not enough** for cold-start evaluation: with a random
split, the same users and items usually appear in both training and test, so the test
set never actually contains a "new" user or item. To evaluate cold start honestly,
we need test sets where the users/items are guaranteed to be unseen during training.

We build four evaluation datasets:

| Split | Purpose |
|---|---|
| A. Standard train/val/test | general model development |
| B. New-user cold-start test | users completely absent from training |
| C. New-item cold-start test | items completely absent from training |
| D. Sparse-user test (optional) | users with only 1–5 interactions (partial cold start) |


### 14A. Standard interaction split (70 / 15 / 15)

If timestamps exist we split chronologically (train on the past, test on the future),
which is more realistic. Otherwise we use a reproducible random split.

Note: the cold-start users/items are removed **first** (sections 14B/14C select them),
so we do this split after carving those out. To keep the notebook readable we select
the cold-start users and items now, then split the remainder.

In [ ]:
if interactions is not None:
    # --- select cold-start USERS (14B) ---
    # COLD_USER_FRAC is set in the configuration cell (easy to change)
    cold_users = select_cold_start_users(interactions, COLD_USER_FRAC, SEED)

In [ ]:
if interactions is not None:
    # --- select cold-start ITEMS (14C), sampled separately for movies and books ---
    # COLD_ITEM_FRAC is set in the configuration cell (easy to change)
    cold_items = select_cold_start_items(interactions, COLD_ITEM_FRAC, SEED)

In [ ]:
if interactions is not None:
    # --- carve out the cold-start test sets ---
    is_cold_user = interactions["user_id"].isin(cold_users)
    is_cold_item = interactions["item_id"].isin(cold_items)

    cold_start_user_test = interactions[is_cold_user & ~is_cold_item].copy()
    cold_start_item_test = interactions[is_cold_item].copy()

    # everything else goes into the standard split
    remaining = interactions[~is_cold_user & ~is_cold_item].copy()

    print("Cold-start user test:", cold_start_user_test.shape)
    print("Cold-start item test:", cold_start_item_test.shape)
    print("Remaining for standard split:", remaining.shape)

In [ ]:
if interactions is not None:
    # --- 70 / 15 / 15 standard split ---
    train_interactions, validation_interactions, test_interactions = split_train_val_test(
        remaining, HAS_TIMESTAMP, SEED, train_frac=0.70, val_frac=0.15)

    print("Train:", train_interactions.shape)
    print("Validation:", validation_interactions.shape)
    print("Test:", test_interactions.shape)

### 14D. Optional sparse-user evaluation group

Users with only **1–5 interactions in the training set** are a *partial* cold-start
scenario: the model has seen them, but barely. This is different from the completely
unseen users in 14B, and it's useful for measuring how quickly the system improves
as a user provides their first few ratings.

In [ ]:
if interactions is not None:
    train_user_counts = train_interactions.groupby("user_id").size()
    sparse_users = set(train_user_counts[train_user_counts.between(1, 5)].index)

    # their test-time interactions come from the standard test set
    sparse_user_test = test_interactions[test_interactions["user_id"].isin(sparse_users)].copy()

    print(f"Sparse users (1-5 training interactions): {len(sparse_users)}")
    print("Sparse-user test set:", sparse_user_test.shape)

## 15. Prevent data leakage

Simple, visible checks that the splits actually do what we claim:

- cold-start users never appear in training
- cold-start items never appear in training
- no identical rows shared between training and validation/test
- cold-start items still have their content metadata in the item table
  (that's what a content-based / generative model will use)

Also worth stating: all cleaning decisions above (dropping duplicates, fixing years,
filling blanks) used only per-row information or the full raw data — nothing was
tuned by peeking at test-set performance, so preprocessing itself does not leak.

In [ ]:
if interactions is not None:
    train_users = set(train_interactions["user_id"])
    train_items = set(train_interactions["item_id"])

    # 1. cold-start users absent from training
    assert len(set(cold_start_user_test["user_id"]) & train_users) == 0, \
        "LEAK: cold-start users found in training!"

    # 2. cold-start items absent from training
    assert len(set(cold_start_item_test["item_id"]) & train_items) == 0, \
        "LEAK: cold-start items found in training!"

    # 3. no duplicated user-item pairs between train and val/test
    train_pairs = set(zip(train_interactions["user_id"], train_interactions["item_id"]))
    val_pairs = set(zip(validation_interactions["user_id"], validation_interactions["item_id"]))
    test_pairs = set(zip(test_interactions["user_id"], test_interactions["item_id"]))
    print("Train/val overlapping user-item pairs:", len(train_pairs & val_pairs))
    print("Train/test overlapping user-item pairs:", len(train_pairs & test_pairs))

    # 4. content metadata still available for cold-start items
    missing_meta = cold_items - set(items["item_id"])
    assert len(missing_meta) == 0, "Some cold-start items are missing from the item table!"

    print("\nAll leakage checks passed.")

## 16. Summarize the final splits

In [ ]:
if interactions is not None:
    split_table = pd.DataFrame([
        split_summary(train_interactions, "train", len(interactions)),
        split_summary(validation_interactions, "validation", len(interactions)),
        split_summary(test_interactions, "test", len(interactions)),
        split_summary(cold_start_user_test, "cold_start_user_test", len(interactions)),
        split_summary(cold_start_item_test, "cold_start_item_test", len(interactions)),
        split_summary(sparse_user_test, "sparse_user_test", len(interactions)),
    ])
    display(split_table)

## 17. Save the split files

Everything the modeling team needs, saved to the project `outputs/` folder (`../outputs/` relative to this notebook).

In [ ]:
out_dir = OUTPUT_DIR   # project outputs folder, set in the configuration cell
os.makedirs(out_dir, exist_ok=True)

if interactions is not None:
    train_interactions.to_csv(os.path.join(out_dir, "train_interactions.csv"), index=False)
    validation_interactions.to_csv(os.path.join(out_dir, "validation_interactions.csv"), index=False)
    test_interactions.to_csv(os.path.join(out_dir, "test_interactions.csv"), index=False)
    cold_start_user_test.to_csv(os.path.join(out_dir, "cold_start_user_test.csv"), index=False)
    cold_start_item_test.to_csv(os.path.join(out_dir, "cold_start_item_test.csv"), index=False)
    sparse_user_test.to_csv(os.path.join(out_dir, "sparse_user_test.csv"), index=False)

print("Saved files:")
for f in sorted(os.listdir(out_dir)):
    print(" -", f)

## 18. Final summary

**What this notebook did:**

- **Cleaning:** removed exact duplicates and rows missing an ID or title from the movie
  and book data; standardized text fields; converted years and ratings to numeric;
  treated placeholder strings ("unknown", "n/a", etc.) as missing; flagged (but did not
  automatically delete) unrealistic publication years and out-of-scale ratings.
- **Standardization:** both catalogs were mapped to one schema
  (`item_id`, `item_type`, `title`, `creator`, `genre`, `description`, `release_year`)
  with `movie_` / `book_` ID prefixes to prevent collisions.
- **Content text:** a `content_text` column combines title, type, genre, creator and
  description per item — this is the input for content-based / generative approaches
  to new-item cold start.
- **Splits:** interactions were divided into a 70/15/15 train/validation/test split
  (chronological when timestamps exist, otherwise seeded random), plus a **new-user**
  cold-start test set (~10% of eligible users, fully removed from training), a
  **new-item** cold-start test set (~10% of eligible movies and books, sampled
  separately per type), and a **sparse-user** test group (users with 1–5 training
  interactions).
- **Leakage checks:** assertions confirm cold-start users/items are absent from
  training and that cold-start items keep their metadata.
- **Outputs:** all tables saved as CSVs in `outputs/` for the modeling team.

**Assumptions made (confirm with the team):**

- Movie and book IDs in the ratings file can be matched by simple membership lookup
  (a raw ID belongs to exactly one catalog).
- The rating scale is 1–5 (change `expected_min` / `expected_max` if not).
- Users/items need ≥ 2 interactions to be eligible for the cold-start samples.
- Publication years outside 1400–2026 are data errors.

---

### Variables / columns to confirm after loading the real datasets

| Variable | Current placeholder | Check against |
|---|---|---|
| `movie_path`, `book_path`, `ratings_path` | resolved to files in `data/raw/` | actual file locations |
| `movie_id_col` | `show_id` | movie file columns |
| `movie_genre_col` | `listed_in` | movie file columns |
| `movie_director_col`, `movie_cast_col`, `movie_desc_col`, `movie_year_col` | see §6 | movie file columns |
| `book_id_col` | `isbn` | book file columns |
| `book_author_col`, `book_genre_col`, `book_desc_col`, `book_year_col` | see §6 | book file columns |
| `user_col`, `rating_item_col`, `rating_col`, `timestamp_col` | see §6 | ratings file columns |
| `expected_min`, `expected_max` | 1, 5 | real rating scale |
| `COLD_USER_FRAC`, `COLD_ITEM_FRAC` | 0.10 | team decision |